# SAE and RL Training Data Analysis
This notebook analyzes the three sets of SAE data metrics: three SAE training variants (BatchTopK, TopK, and Threshold/L1).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
# In local mode, assuming we are running inside the folder or have the paths correct.
# If on Colab, you might need to adjust the paths below.
import os
import sys
import pandas as pd

# Check if we are running in Google Colab
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    # Update this path to where your files are stored in Google Drive
    base_path = '/content/drive/MyDrive/sae_collation'
# Check if we're running from workspace root or inside All_Data locally
elif os.path.exists("All_Data"):
    base_path = "All_Data"
else:
    base_path = "."

path_batchtopk = os.path.join(base_path, "SAE_Collation - sae_collation_batchtopk.csv")
path_threshold = os.path.join(base_path, "SAE_Collation - sae_collation_threshold.csv")
path_topk = os.path.join(base_path, "SAE_Collation - sae_collation_topk.csv")

df_batchtopk = pd.read_csv(path_batchtopk) if os.path.exists(path_batchtopk) else pd.DataFrame()
df_threshold = pd.read_csv(path_threshold) if os.path.exists(path_threshold) else pd.DataFrame()
df_topk = pd.read_csv(path_topk) if os.path.exists(path_topk) else pd.DataFrame()

print(f"BatchTopK Data: {df_batchtopk.shape}")
print(f"Threshold/L1 Data: {df_threshold.shape}")


## 2. Unifying Data (Pareto Metrics)


In [ ]:
plot_data = []

if not df_batchtopk.empty:
    df_j = df_batchtopk.copy()
    arch = df_j.get('Architecture', 'BatchTopK').astype(str)
    df_j['Source'] = 'BatchTopK (' + arch + ')'
    df_j['L0'] = df_j['Avg L0']
    df_j['NMSE'] = df_j['Recon Loss (NMSE)']
    df_j['Dead_Latent_Frac'] = df_j['Dead Latents %'] / 100.0
    plot_data.append(df_j[['Source', 'L0', 'NMSE', 'Expansion', 'Dead_Latent_Frac']])

if not df_topk.empty:
    df_m = df_topk.copy()
    arch = df_m.get('Architecture', 'TopK').astype(str)
    df_m['Source'] = 'TopK (' + arch + ')'
    df_m['L0'] = df_m['Avg L0']
    df_m['NMSE'] = df_m['Recon Loss (NMSE)']
    if 'Dead Latents %' in df_m.columns:
        df_m['Dead_Latent_Frac'] = df_m['Dead Latents %'] / 100.0
    else:
        df_m['Dead_Latent_Frac'] = np.nan
    plot_data.append(df_m[['Source', 'L0', 'NMSE', 'Expansion', 'Dead_Latent_Frac']])

if not df_threshold.empty:
    df_t = df_threshold.copy()
    df_t['Source'] = 'Threshold/L1'
    df_t['L0'] = df_t['hard_l0']
    # This set has 'explained_var' which is ~ 1 - NMSE. We convert it to NMSE to normalize against the other two sets!
    df_t['NMSE'] = 1.0 - df_t['explained_var']
    df_t['Dead_Latent_Frac'] = df_t['dead_frac_batch']
    # Convert expansion string to int if possible
    if df_t['expansion_factor'].dtype == object:
        df_t['Expansion'] = df_t['expansion_factor'].str.replace('x', '').astype(float)
    else:
        df_t['Expansion'] = df_t['expansion_factor']
    plot_data.append(df_t[['Source', 'L0', 'NMSE', 'Expansion', 'Dead_Latent_Frac']])

if plot_data:
    df_all = pd.concat(plot_data, ignore_index=True)
    print("Unified DataFrame preview:\n", df_all.head())
else:


In [ ]:
if not df_all.empty:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_all, x="L0", y="NMSE", hue="Source", style="Expansion", s=100, alpha=0.8)
    plt.title("Architecture Comparison: NMSE vs L0 Pareto Frontier")
    plt.xlabel("L0 (Average active latents)")
    plt.ylabel("Normalized Mean Squared Error (NMSE)")
    plt.yscale('log')
    plt.xscale('log')
    plt.tight_layout()


In [ ]:
if not df_all.empty:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df_all, x="L0", y="Dead_Latent_Frac", hue="Source", style="Expansion", s=100, alpha=0.8)
    plt.title("Dead Latents vs L0")
    plt.xlabel("L0 (Average active latents)")
    plt.ylabel("Dead Latent Fraction")
    plt.xscale('log')
    plt.tight_layout()


In [ ]:
if not df_threshold.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.scatterplot(data=df_threshold, x="features_gt_10pct_firing", y="explained_var", hue="expansion_factor", ax=axes[0], s=80)
    axes[0].set_title("Explained Var vs highly active features")
    
    sns.lineplot(data=df_threshold.sort_values("hard_l0"), x="hard_l0", y="explained_var", hue="expansion_factor", marker="o", ax=axes[1])
    axes[1].set_title("Explained Variance Pareto")
    plt.tight_layout()


In [ ]:
if not df_batchtopk.empty:
    print("\nBatchTopK Data By Layer:")
    display(df_batchtopk.groupby('Layer')[['Avg L0', 'Recon Loss (MSE)']].mean())
    
if not df_topk.empty:
    print("\nTopK Data By Chain:")
    display(df_topk.groupby('Chain')[['Avg L0', 'Recon Loss (MSE)']].mean())


## 6. Deep Checkpoint Analysis (Weight Norms, Birth/Death, CKA)


In [ ]:
import os
import sys
import glob
import torch
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

device = "cuda" if torch.cuda.is_available() else "cpu"

# Clone the HF repo to get the .pt files if they don't exist locally
if False:
    pass # Using local paths instead of huggingface

# Check if we are running in Google Colab
if 'google.colab' in sys.modules:
    # Update this path to where the SAE checkpoint weights are stored in Google Drive
    base_path = '/content/drive/MyDrive/sae-rl-qwen05b-layers'   
elif os.path.exists("sae-rl-qwen05b-layers"):
    base_path = "sae-rl-qwen05b-layers"
else:
    base_path = "../sae-rl-qwen05b-layers"

layer_dirs = ["layer6", "layer12", "layer18", "layer23"]

all_results = []
print("Extracting statistics from checkpoint weights...\n")
for layer in layer_dirs:
    layer_path = os.path.join(base_path, layer)
    if os.path.exists(layer_path):
        pt_files = glob.glob(os.path.join(layer_path, "*.pt"))
        for pt_file in pt_files:
            file_name = os.path.basename(pt_file)
            ckpt = torch.load(pt_file, map_location="cpu", weights_only=False)
            row = {'layer': layer, 'file_name': file_name}
            
            if "step" in file_name:
                try:
                    row['step'] = int(file_name.split("step")[1].split("_")[0].split(".")[0])
                    row['model_type'] = "ppo"
                except:
                    row['step'] = None
            else:
                row['step'] = 0
                row['model_type'] = "base"
                
            state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
            row['encoder_weight_norm'] = state_dict.get('encoder.weight', state_dict.get('W_enc', torch.tensor(0.0))).norm().item() if any(k in state_dict for k in ['encoder.weight', 'W_enc']) else 0.0
            row['decoder_weight_norm'] = state_dict.get('decoder.weight', state_dict.get('W_dec', torch.tensor(0.0))).norm().item() if any(k in state_dict for k in ['decoder.weight', 'W_dec']) else 0.0
            all_results.append(row)

combined_df = pd.DataFrame(all_results)
if not combined_df.empty:
    combined_df = combined_df.sort_values(by=['layer', 'step']).reset_index(drop=True)


## 6. Deep CSV Metrics Analysis Across Variants


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="darkgrid")

plt.figure(figsize=(12, 6))

if not df_topk.empty:
    y_col = "Recon Loss (MSE)" if not df_topk["Recon Loss (MSE)"].dropna().empty else "Recon Loss (NMSE)"
    sns.scatterplot(data=df_topk, x="Avg L0", y=y_col, hue="Chain", style="Layer", s=100, palette="viridis")
    plt.title(f"TopK: {y_col} vs L0 Grouped by Chain and Layer")
    plt.xscale('log')
    plt.yscale('log')
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
    plt.tight_layout()


In [ ]:
if not df_threshold.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.scatterplot(data=df_threshold, x="hard_l0", y="explained_var", hue="expansion_factor", ax=axes[0], s=80, palette="magma")
    axes[0].set_title("Threshold/L1: Explained Variance vs Hard L0")
    
    sns.scatterplot(data=df_threshold, x="hard_l0", y="dead_frac_batch", hue="expansion_factor", ax=axes[1], s=80, palette="magma")
    axes[1].set_title("Threshold/L1: Dead Fraction vs Hard L0")
    
    sns.scatterplot(data=df_threshold, x="mean_threshold", y="fire_rate_mean", hue="expansion_factor", ax=axes[2], s=80, palette="magma")
    axes[2].set_title("Threshold/L1: Mean Fire Rate vs Threshold")
    
    plt.tight_layout()


In [ ]:
if not df_topk.empty:
    if not df_topk["Dead Latents %"].dropna().empty:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df_topk, x="Layer", y="Dead Latents %", hue="Chain")
        plt.title("TopK: Dead Latents Percentage Across Layers and Chains")
        plt.show()
    else:


In [ ]:
if not df_batchtopk.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_batchtopk, x="Layer", y="Frac Rec %", hue="K", palette="Blues_d")
    plt.title("BatchTopK: Fractional Recovery % by Layer and K-value")
    plt.show()
    
    plt.figure(figsize=(10, 5))
    sns.scatterplot(data=df_batchtopk, x="Avg L0", y="Recon Loss (MSE)", hue="Layer", size="K", sizes=(50, 200), palette="coolwarm")
    plt.title("BatchTopK: MSE vs L0 Tradeoff (Bubble size = K)")
    plt.xscale('log')
    plt.yscale('log')


In [ ]:
if not combined_df.empty:
    summary_stats = combined_df.groupby(['layer', 'model_type']).agg(
        avg_encoder_norm=('encoder_weight_norm', 'mean'),
        avg_decoder_norm=('decoder_weight_norm', 'mean'),
        max_step=('step', 'max')
    ).reset_index()
    print("Summary of Norms by Layer:")
    display(summary_stats)

    base_models = combined_df[combined_df['model_type'] == 'base'].set_index('layer')
    ppo_models_drift = combined_df[combined_df['model_type'] == 'ppo'].copy()
    
    def get_base_norm(layer_name):
        return base_models.loc[layer_name, 'encoder_weight_norm'] if layer_name in base_models.index else None

    ppo_models_drift['base_encoder_norm'] = ppo_models_drift['layer'].apply(get_base_norm)
    ppo_models_drift['encoder_norm_diff'] = ppo_models_drift['encoder_weight_norm'] - ppo_models_drift['base_encoder_norm']
    display(ppo_models_drift[['layer', 'step', 'encoder_weight_norm', 'encoder_norm_diff']].head())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=ppo_models_drift, x="step", y="encoder_weight_norm", hue="layer", marker="o", ax=axes[0])
    axes[0].set_title('Encoder Weight Norm across RL Steps')
    
    sns.barplot(data=ppo_models_drift, x="layer", y="encoder_norm_diff", hue="layer", errorbar='sd', ax=axes[1])
    axes[1].set_title('Encoder Drift (Norm Difference vs Base Model)')
    plt.tight_layout()


In [ ]:
BIRTH_DEATH_THRESHOLD = 0.7

def get_normalized_decoder_features(state_dict):
    if 'decoder.weight' in state_dict:
        W_dec = state_dict['decoder.weight']
    elif 'W_dec' in state_dict:
        W_dec = state_dict['W_dec']
    else:
        return None
    return torch.nn.functional.normalize(W_dec, p=2, dim=0).cpu().numpy()

def match_features_hungarian(sim_matrix, threshold=BIRTH_DEATH_THRESHOLD):
    cost_matrix = 1.0 - np.abs(sim_matrix) 
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matched_sims = sim_matrix[row_ind, col_ind]
    good_matches = np.abs(matched_sims) >= threshold
    n_matched = good_matches.sum()
    n_features_a, n_features_b = sim_matrix.shape
    return {
        "n_matched": int(n_matched),
        "n_births": len(set(range(n_features_b)) - set(col_ind[good_matches])),
        "n_deaths": len(set(range(n_features_a)) - set(row_ind[good_matches])),
        "mean_matched_sim": float(np.abs(matched_sims[good_matches]).mean()) if n_matched > 0 else 0.0
    }

birth_death_results = []
for layer in layer_dirs:
    layer_files = combined_df[combined_df['layer'] == layer].sort_values(by='step') if not combined_df.empty else []
    if len(layer_files) <= 1: continue
    
    W_dec_a = get_normalized_decoder_features(torch.load(os.path.join(base_path, layer, layer_files.iloc[0]['file_name']), map_location="cpu", weights_only=False).get('state_dict', {}))
    if W_dec_a is None: continue
        
    for i in range(1, len(layer_files)):
        curr_file_row = layer_files.iloc[i]
        W_dec_b = get_normalized_decoder_features(torch.load(os.path.join(base_path, layer, curr_file_row['file_name']), map_location="cpu", weights_only=False).get('state_dict', {}))
        if W_dec_b is not None:
            match_result = match_features_hungarian(W_dec_a.T @ W_dec_b)
            result = {"layer": layer, "step_to": curr_file_row['step'], "n_births": match_result['n_births'], "n_deaths": match_result['n_deaths'], "n_matched": match_result['n_matched']}
            birth_death_results.append(result)
            W_dec_a = W_dec_b

if birth_death_results:
    birth_death_df = pd.DataFrame(birth_death_results)
    layer_df = birth_death_df[birth_death_df['layer'] == 'layer18']
    if not layer_df.empty:
        plt.figure(figsize=(10, 4))
        plt.plot(layer_df["step_to"], layer_df["n_births"], "o-", label="Births", color="green")
        plt.plot(layer_df["step_to"], layer_df["n_deaths"], "s-", label="Deaths", color="red")
        plt.plot(layer_df["step_to"], layer_df["n_matched"], "^-", label="Persisted", color="blue")
        plt.title("Feature Birth & Death (Layer 18)")
        plt.legend()


In [ ]:
def linear_CKA(X, Y):
    if X.shape != Y.shape: return 0.0
    X, Y = X - X.mean(axis=0), Y - Y.mean(axis=0)
    denom = np.linalg.norm(X.T @ X, "fro") * np.linalg.norm(Y.T @ Y, "fro")
    return float(np.linalg.norm(Y.T @ X, "fro") ** 2 / denom) if denom != 0 else 0.0

if not combined_df.empty:
    layer_files = combined_df[combined_df['layer'] == 'layer18'].sort_values(by='step')
    W_decs, ordered = {}, []
    for _, row in layer_files.iterrows():
        W_dec = get_normalized_decoder_features(torch.load(os.path.join(base_path, "layer18", row['file_name']), map_location="cpu", weights_only=False).get('state_dict', {}))
        if W_dec is not None:
            W_decs[f"Step {row['step']}"] = W_dec
            ordered.append(f"Step {row['step']}")
            
    n = len(ordered)
    if n > 1:
        cka_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(i, n):
                cka_matrix[i, j] = cka_matrix[j, i] = linear_CKA(W_decs[ordered[i]], W_decs[ordered[j]])
        
        plt.figure(figsize=(6, 5))
        im = plt.imshow(cka_matrix, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
        plt.xticks(range(n), ordered, rotation=45, ha="right")
        plt.yticks(range(n), ordered)
        plt.title("CKA Matrix (Layer 18 SAE Decoders)")
        plt.colorbar(im)


In [ ]:
def plot_weight_norm_distributions(layer_folder):
    layer_df = combined_df[combined_df['layer'] == layer_folder].sort_values('step') if not combined_df.empty else []
    if len(layer_df) == 0: return
    plt.figure(figsize=(8, 5))
    colors = sns.color_palette("viridis", n_colors=len(layer_df))
    for i, (_, row) in enumerate(layer_df.iterrows()):
        state_dict = torch.load(os.path.join(base_path, layer_folder, row['file_name']), map_location='cpu').get('state_dict', {})
        w_dec = state_dict.get("W_dec", state_dict.get("decoder.weight", {}))
        if hasattr(w_dec, 'T'):
            if 'decoder.weight' in state_dict: w_dec = w_dec.T
            norms = torch.norm(w_dec.float(), p=2, dim=-1)
            sns.kdeplot(norms.numpy(), color=colors[i], label=f"Step {row['step']}", alpha=0.7)
    plt.title(f"Decoder Norm Distribution ({layer_folder})")
    plt.legend()
    plt.show()



In [ ]:
def plot_mutual_orthogonality(layer_folder):
    layer_df = combined_df[combined_df['layer'] == layer_folder].sort_values('step') if not combined_df.empty else []
    if len(layer_df) == 0: return
    steps, orthos = [], []
    for i, (_, row) in enumerate(layer_df.iterrows()):
        state_dict = torch.load(os.path.join(base_path, layer_folder, row['file_name']), map_location='cpu').get('state_dict', {})
        w_dec = state_dict.get("W_dec", state_dict.get("decoder.weight", {}))
        if hasattr(w_dec, 'float'):
            if 'decoder.weight' in state_dict: w_dec = w_dec.T
            w_dec = w_dec.float()
            w_norm = w_dec / torch.norm(w_dec, p=2, dim=-1, keepdim=True)
            indices = torch.randperm(w_norm.shape[0])[:min(2000, w_norm.shape[0])]
            w_sample = w_norm[indices]
            sim_matrix = torch.matmul(w_sample, w_sample.T)
            sim_matrix.fill_diagonal_(0)
            steps.append(row['step'])
            orthos.append(torch.mean(torch.abs(sim_matrix)).item())
            
    if steps:
        plt.figure(figsize=(7, 4))
        plt.plot(steps, orthos, marker='o', color='purple')
        plt.title(f"Mutual Orthogonality/Entanglement ({layer_folder})")
        plt.show()

